In [67]:
import numpy as np
import pandas as pd
import gzip
import json
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer #FOR TF-IDF (FINDING IMPORTANT WORDS)
from sklearn.neighbors import BallTree #FOR SPATIAL ANALYSIS (NEIGHBOR DENSITY)

# Generalized new reviews analysis approach for multiple states (added suburban + restaurant/non-restaurant subcategories)

In [68]:

def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.loads(l)

def parse_first_n(path, n=1000000):
    g = gzip.open('data/' + path, 'r')
    for i, l in enumerate(g):
        if i >= n:
            break
        yield json.loads(l)

state_reviews = pd.DataFrame(parse_first_n("review-New_York.json.gz"))
state_meta = pd.DataFrame(parse("data/meta-New_York.json.gz"))

## CLASSIFYING AREA PART

In [69]:
state_merged_reviews = state_meta.merge(state_reviews, on='gmap_id')
coords = state_merged_reviews[['latitude', 'longitude']].dropna().values
coords_rad = np.radians(coords)
df_valid = state_merged_reviews[['latitude','longitude']].dropna().copy()
coords_rad = np.radians(df_valid[['latitude','longitude']].values)

tree = BallTree(coords_rad, metric='haversine')
radius_miles = 5
radius = radius_miles / 3959
counts = tree.query_radius(coords_rad, r=radius, count_only=True)

df_valid['local_density'] = counts - 1
state_merged_reviews.loc[df_valid.index, 'local_density'] = df_valid['local_density']
state_merged_reviews['density_bin'] = pd.qcut(
    state_merged_reviews['local_density'],
    q=3,
    labels=['Rural','Suburban','Urban']
)
state_merged_reviews = state_merged_reviews[~state_merged_reviews.get('text').isna()]

In [70]:
## Maps of Important Words
restaurant_dict = {}
nonrestaurant_dict = {}

### URBAN

In [71]:
state_urban_reviews = state_merged_reviews[state_merged_reviews.get('density_bin') == 'Urban']
urban_state_reviews_good = state_urban_reviews[(state_urban_reviews.get('rating') == 4) | (state_urban_reviews.get('rating') == 5)]
urban_state_reviews_bad = state_urban_reviews[(state_urban_reviews.get('rating') == 1) | (state_urban_reviews.get('rating') == 2)]


#### Restaurant Good

In [72]:
state_urban_restaurant_reviews_good = urban_state_reviews_good[urban_state_reviews_good['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_urban_state_restaurant_reviews_good = state_urban_restaurant_reviews_good.get('text').tolist()

rest_urban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_urban_result_good = rest_urban_tfidf_good.fit_transform(list_of_urban_state_restaurant_reviews_good)

rest_urban_feature_names_good = rest_urban_tfidf_good.get_feature_names_out()
rest_mean_urban_tfidf_good = np.asarray(rest_urban_result_good.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_good_urban = rest_mean_urban_tfidf_good.argsort()[-top_n:][::-1]

rest_top_words_good_urban = [(rest_urban_feature_names_good[i], rest_mean_urban_tfidf_good[i]) for i in rest_top_indices_good_urban]

rows = []

for word, score in rest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Restaurant",
        "review_group": "Good"
    })

#### Non-Restaurant Good

In [73]:
state_urban_nonrestaurant_reviews_good = urban_state_reviews_good[
    urban_state_reviews_good['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_urban_state_nonrestaurant_reviews_good = state_urban_nonrestaurant_reviews_good.get('text').tolist()

nonrest_urban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_urban_result_good = nonrest_urban_tfidf_good.fit_transform(list_of_urban_state_nonrestaurant_reviews_good)

nonrest_urban_feature_names_good = nonrest_urban_tfidf_good.get_feature_names_out()
nonrest_mean_urban_tfidf_good = np.asarray(nonrest_urban_result_good.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_good_urban = nonrest_mean_urban_tfidf_good.argsort()[-top_n:][::-1]

nonrest_top_words_good_urban = [(nonrest_urban_feature_names_good[i], nonrest_mean_urban_tfidf_good[i]) for i in nonrest_top_indices_good_urban]

for word, score in nonrest_top_words_good_urban:
    nonrestaurant_dict[word] = score
    print(word, score)

for word, score in nonrest_top_words_good_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Non-Restaurant",
        "review_group": "Good"
    })


translated google 0.011968911363859148
highly recommend 0.010629393955215824
great service 0.009084781591983525
customer service 0.007506829100414537
great place 0.005151246081975018
great experience 0.004995285567719902
excellent service 0.004554650547349017
good service 0.004535777296553024
friendly staff 0.0037919814556546237
highly recommended 0.003749965548661929
definitely recommend 0.0034571855155250894
great customer 0.0032074391384493235
great job 0.003041263389732314
google good 0.003017962033582857
love place 0.002976411025645177
new york 0.0028314688200516648
staff friendly 0.0028043448588085137
service great 0.0023758783471157455
nice place 0.002253365319544983
great staff 0.002244938912181026


#### Restaurant Bad

In [74]:
state_urban_restaurant_reviews_bad = urban_state_reviews_bad[urban_state_reviews_bad['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_urban_state_restaurant_reviews_bad = state_urban_restaurant_reviews_bad.get('text').tolist()

rest_urban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_urban_result_bad = rest_urban_tfidf_bad.fit_transform(list_of_urban_state_restaurant_reviews_bad)

rest_urban_feature_names_bad = rest_urban_tfidf_bad.get_feature_names_out()
rest_mean_urban_tfidf_bad = np.asarray(rest_urban_result_bad.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_bad_urban = rest_mean_urban_tfidf_bad.argsort()[-top_n:][::-1]

rest_top_words_bad_urban = [(rest_urban_feature_names_bad[i], rest_mean_urban_tfidf_bad[i]) for i in rest_top_indices_bad_urban]

for word, score in rest_top_words_bad_urban:
    restaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_bad_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Restaurant",
        "review_group": "Bad"
    })

translated google 0.03915895271733165
customer service 0.01620869327333273
food good 0.011032707422172574
don know 0.007729604173546768
bad service 0.007645013071812362
tasted like 0.007245369154195354
waste money 0.006634853017543879
food ok 0.005691788601206116
30 minutes 0.005595392422618092
poor service 0.0054742096629610115
good food 0.005205009977262946
terrible service 0.0047982478216200344
food poisoning 0.004776575436788856
horrible service 0.0046441943491638375
waste time 0.00461531223642633
long time 0.004540347078477334
20 minutes 0.004479838730557086
really bad 0.004455518056788763
new york 0.004422470997100071
order food 0.004413057233023041


#### Non-Restaurant Bad

In [75]:
state_urban_nonrestaurant_reviews_bad = urban_state_reviews_bad[
    urban_state_reviews_bad['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_urban_state_nonrestaurant_reviews_bad = state_urban_nonrestaurant_reviews_bad.get('text').tolist()

nonrest_urban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_urban_result_bad = nonrest_urban_tfidf_bad.fit_transform(list_of_urban_state_nonrestaurant_reviews_bad)

nonrest_urban_feature_names_bad = nonrest_urban_tfidf_bad.get_feature_names_out()
nonrest_mean_urban_tfidf_bad = np.asarray(nonrest_urban_result_bad.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_bad_urban = nonrest_mean_urban_tfidf_bad.argsort()[-top_n:][::-1]

nonrest_top_words_bad_urban = [(nonrest_urban_feature_names_bad[i], nonrest_mean_urban_tfidf_bad[i]) for i in nonrest_top_indices_bad_urban]

for word, score in nonrest_top_words_bad_urban:
    nonrestaurant_dict[word] = -score
    print(word, score)

for word, score in nonrest_top_words_bad_urban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Urban",
        "business_type": "Non-Restaurant",
        "review_group": "Bad"
    })

customer service 0.014816170993273274
translated google 0.01164444582686232
waste time 0.0058417605059159616
stay away 0.005361984394535189
don know 0.004590670051877051
bad service 0.004524770646421948
horrible service 0.0029744743446486668
terrible service 0.002872723996890426
extremely rude 0.0028072327480387895
bad experience 0.002788969170553399
worst experience 0.002775896944785128
worst customer 0.002733486926281023
don waste 0.0027065390730514538
worst place 0.0026547064467683642
new york 0.0024823738023392024
horrible customer 0.002445915520686237
answer phone 0.002393631208483572
credit card 0.002385921172059372
don care 0.0023433157750277368
don want 0.0023147800746907764


### SUBURBAN

In [76]:
state_suburban_reviews = state_merged_reviews[state_merged_reviews.get('density_bin') == 'Suburban']
suburban_state_reviews_good = state_suburban_reviews[(state_suburban_reviews.get('rating') == 4) | (state_suburban_reviews.get('rating') == 5)]
suburban_state_reviews_bad = state_suburban_reviews[(state_suburban_reviews.get('rating') == 1) | (state_suburban_reviews.get('rating') == 2)]


#### Restaurant Good

In [77]:
state_suburban_restaurant_reviews_good = suburban_state_reviews_good[suburban_state_reviews_good['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_suburban_state_restaurant_reviews_good = state_suburban_restaurant_reviews_good.get('text').tolist()

rest_suburban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_suburban_result_good = rest_suburban_tfidf_good.fit_transform(list_of_suburban_state_restaurant_reviews_good)

rest_suburban_feature_names_good = rest_suburban_tfidf_good.get_feature_names_out()
rest_mean_suburban_tfidf_good = np.asarray(rest_suburban_result_good.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_good_suburban = rest_mean_suburban_tfidf_good.argsort()[-top_n:][::-1]

rest_top_words_good_suburban = [(rest_suburban_feature_names_good[i], rest_mean_suburban_tfidf_good[i]) for i in rest_top_indices_good_suburban]

for word, score in rest_top_words_good_suburban:
    restaurant_dict[word] = score
    print(word, score)

for word, score in rest_top_words_good_suburban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Restaurant",
        "review_group": "Good"
    })


good food 0.026741244673126607
great food 0.025538155925447933
translated google 0.024637015877325953
food great 0.01662947730499607
food good 0.013550423287173118
great service 0.011300908060170069
customer service 0.010961239155634472
delicious food 0.008859799438692629
great place 0.008824235086518317
food delicious 0.008436562039554483
food service 0.008429737499165401
highly recommend 0.00838198419796875
really good 0.008023536550609838
good service 0.007941909970304578
friendly staff 0.007189038114970412
food amazing 0.0069889469395468945
love place 0.0068121289121386396
amazing food 0.006696554580105868
chinese food 0.006419859932968573
excellent food 0.00639745492361332


#### Non-Restaurant Good

In [78]:
state_suburban_nonrestaurant_reviews_good = suburban_state_reviews_good[
    suburban_state_reviews_good['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_suburban_state_nonrestaurant_reviews_good = state_suburban_nonrestaurant_reviews_good.get('text').tolist()

nonrest_suburban_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_suburban_result_good = nonrest_suburban_tfidf_good.fit_transform(list_of_suburban_state_nonrestaurant_reviews_good)

nonrest_suburban_feature_names_good = nonrest_suburban_tfidf_good.get_feature_names_out()
nonrest_mean_suburban_tfidf_good = np.asarray(nonrest_suburban_result_good.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_good_suburban = nonrest_mean_suburban_tfidf_good.argsort()[-top_n:][::-1]

nonrest_top_words_good_suburban = [(nonrest_suburban_feature_names_good[i], nonrest_mean_suburban_tfidf_good[i]) for i in nonrest_top_indices_good_suburban]

for word, score in nonrest_top_words_good_suburban:
    nonrestaurant_dict[word] = score
    print(word, score)

for word, score in nonrest_top_words_good_suburban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Non-Restaurant",
        "review_group": "Good"
    })

highly recommend 0.01169360695096396
great service 0.010101826800472372
customer service 0.008370519908824156
translated google 0.006615270419806889
great place 0.006131535463086292
great experience 0.005393049585619029
excellent service 0.005247510287708378
great job 0.004758919688034466
friendly staff 0.0040705372847685784
great customer 0.003998610564684425
good service 0.003996594221289961
definitely recommend 0.003966514154457636
highly recommended 0.003665625661254903
staff friendly 0.0032258896570568013
love place 0.0031810546649580357
service great 0.0026803964271062074
great staff 0.002615791953533774
great work 0.0024814872170118045
did great 0.00240297195357017
friendly helpful 0.002337788659185398


#### Restaurant Bad

In [79]:
state_suburban_restaurant_reviews_bad = suburban_state_reviews_bad[suburban_state_reviews_bad['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_suburban_state_restaurant_reviews_bad = state_suburban_restaurant_reviews_bad.get('text').tolist()

rest_suburban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_suburban_result_bad = rest_suburban_tfidf_bad.fit_transform(list_of_suburban_state_restaurant_reviews_bad)

rest_suburban_feature_names_bad = rest_suburban_tfidf_bad.get_feature_names_out()
rest_mean_suburban_tfidf_bad = np.asarray(rest_suburban_result_bad.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_bad_suburban = rest_mean_suburban_tfidf_bad.argsort()[-top_n:][::-1]

rest_top_words_bad_suburban = [(rest_suburban_feature_names_bad[i], rest_mean_suburban_tfidf_bad[i]) for i in rest_top_indices_bad_suburban]

for word, score in rest_top_words_bad_suburban:
    restaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_bad_suburban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Restaurant",
        "review_group": "Bad"
    })

translated google 0.029980275278978674
customer service 0.021054791630622904
food good 0.012119203645304994
waste money 0.009768442199977741
bad service 0.00891494437234856
don know 0.008157278235457342
don waste 0.00616702167343554
chinese food 0.006089752918099024
food horrible 0.005709366828897386
stay away 0.005701199630160637
good food 0.005687765454755548
got food 0.005652981672556491
waste time 0.005469374129057696
fried rice 0.0054019702033840525
tasted like 0.005377175256786917
taste like 0.005204309135489288
terrible service 0.005083195063111719
placed order 0.0050072812419830895
recommend place 0.005004436036110373
service horrible 0.0047586800524769286


#### Non-Restaurant Bad

In [80]:
state_suburban_nonrestaurant_reviews_bad = suburban_state_reviews_bad[
    suburban_state_reviews_bad['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_suburban_state_nonrestaurant_reviews_bad = state_suburban_nonrestaurant_reviews_bad.get('text').tolist()

nonrest_suburban_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_suburban_result_bad = nonrest_suburban_tfidf_bad.fit_transform(list_of_suburban_state_nonrestaurant_reviews_bad)

nonrest_suburban_feature_names_bad = nonrest_suburban_tfidf_bad.get_feature_names_out()
nonrest_mean_suburban_tfidf_bad = np.asarray(nonrest_suburban_result_bad.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_bad_suburban = nonrest_mean_suburban_tfidf_bad.argsort()[-top_n:][::-1]

nonrest_top_words_bad_suburban = [(nonrest_suburban_feature_names_bad[i], nonrest_mean_suburban_tfidf_bad[i]) for i in nonrest_top_indices_bad_suburban]

for word, score in nonrest_top_words_bad_suburban:
    nonrestaurant_dict[word] = -score
    print(word, score)

for word, score in nonrest_top_words_bad_suburban:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Suburban",
        "business_type": "Non-Restaurant",
        "review_group": "Bad"
    })

customer service 0.01581163091061511
translated google 0.007922216787249694
stay away 0.006200344425364573
waste time 0.005070951697481375
don know 0.004230400865727753
bad service 0.003980572393200231
horrible customer 0.003112893439920938
horrible service 0.003000169587397419
worst experience 0.002864929312301942
worst customer 0.002828437524567667
answer phone 0.0027509221609952495
worst place 0.002718693178322455
don waste 0.0026883915285302517
terrible service 0.0026822051404963004
poor customer 0.0026557706518938145
extremely rude 0.0026019800162780478
recommend place 0.002527185535599352
make sure 0.0024748405559988015
zero stars 0.002433493115592058
horrible experience 0.002429821434606376


### RURAL

In [81]:
state_rural_reviews = state_merged_reviews[state_merged_reviews.get('density_bin') == 'Rural']
rural_state_reviews_good = state_rural_reviews[(state_rural_reviews.get('rating') == 4) | (state_rural_reviews.get('rating') == 5)]
rural_state_reviews_bad = state_rural_reviews[(state_rural_reviews.get('rating') == 1) | (state_rural_reviews.get('rating') == 2)]


#### Restaurant Good

In [82]:
state_rural_restaurant_reviews_good = rural_state_reviews_good[rural_state_reviews_good['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_rural_state_restaurant_reviews_good = state_rural_restaurant_reviews_good.get('text').tolist()

rest_rural_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_rural_result_good = rest_rural_tfidf_good.fit_transform(list_of_rural_state_restaurant_reviews_good)

rest_rural_feature_names_good = rest_rural_tfidf_good.get_feature_names_out()
rest_mean_rural_tfidf_good = np.asarray(rest_rural_result_good.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_good_rural = rest_mean_rural_tfidf_good.argsort()[-top_n:][::-1]

rest_top_words_good_rural = [(rest_rural_feature_names_good[i], rest_mean_rural_tfidf_good[i]) for i in rest_top_indices_good_rural]

for word, score in rest_top_words_good_rural:
    restaurant_dict[word] = score
    print(word, score)

for word, score in rest_top_words_good_rural:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Restaurant",
        "review_group": "Good"
    })


great food 0.03304902238224074
good food 0.02078625452066821
food great 0.01727817575541429
great service 0.013447994441510212
great place 0.01086107919921567
food service 0.010471153468654546
food good 0.010449047964099054
friendly staff 0.01043621059110219
great pizza 0.010281905525996714
best pizza 0.010217315952897215
ice cream 0.010124313218482266
excellent food 0.009717090640561641
highly recommend 0.009412233079719574
translated google 0.008758061781144577
really good 0.007561678099015475
customer service 0.007246350948755383
delicious food 0.00721049825106243
food delicious 0.006954327919169065
good pizza 0.006822553483723483
love place 0.006802962859874726


#### Non-Restaurant Good

In [83]:
state_rural_nonrestaurant_reviews_good = rural_state_reviews_good[
    rural_state_reviews_good['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_rural_state_nonrestaurant_reviews_good = state_rural_nonrestaurant_reviews_good.get('text').tolist()

nonrest_rural_tfidf_good = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_rural_result_good = nonrest_rural_tfidf_good.fit_transform(list_of_rural_state_nonrestaurant_reviews_good)

nonrest_rural_feature_names_good = nonrest_rural_tfidf_good.get_feature_names_out()
nonrest_mean_rural_tfidf_good = np.asarray(nonrest_rural_result_good.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_good_rural = nonrest_mean_rural_tfidf_good.argsort()[-top_n:][::-1]

nonrest_top_words_good_rural = [(nonrest_rural_feature_names_good[i], nonrest_mean_rural_tfidf_good[i]) for i in nonrest_top_indices_good_rural]

for word, score in nonrest_top_words_good_rural:
    nonrestaurant_dict[word] = score
    print(word, score)

for word, score in nonrest_top_words_good_rural:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Non-Restaurant",
        "review_group": "Good"
    })

highly recommend 0.011215970390697046
great service 0.009677001208395935
great place 0.008745354978248087
customer service 0.006804021887470367
great job 0.005450673818516621
great experience 0.004812646099373886
excellent service 0.004522900405621857
friendly staff 0.004499897438141479
great people 0.003926452678143721
friendly helpful 0.003313185075518295
love place 0.0032431528868288382
staff friendly 0.0030762793296470153
great customer 0.00303888629878903
great work 0.003015338786776939
nice place 0.003011584620171928
translated google 0.003001606964491695
highly recommended 0.002932595114868115
definitely recommend 0.002872027146522912
great staff 0.0027851291273793865
did great 0.0026834506380285107


#### Restaurant Bad

In [84]:
state_rural_restaurant_reviews_bad = rural_state_reviews_bad[rural_state_reviews_bad['category'].apply(lambda cats: any('restaurant' in c.lower() for c in cats)if isinstance(cats, list) else False)]
list_of_rural_state_restaurant_reviews_bad = state_rural_restaurant_reviews_bad.get('text').tolist()

rest_rural_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

rest_rural_result_bad = rest_rural_tfidf_bad.fit_transform(list_of_rural_state_restaurant_reviews_bad)

rest_rural_feature_names_bad = rest_rural_tfidf_bad.get_feature_names_out()
rest_mean_rural_tfidf_bad = np.asarray(rest_rural_result_bad.mean(axis=0)).flatten()
top_n = 20

rest_top_indices_bad_rural = rest_mean_rural_tfidf_bad.argsort()[-top_n:][::-1]

rest_top_words_bad_rural = [(rest_rural_feature_names_bad[i], rest_mean_rural_tfidf_bad[i]) for i in rest_top_indices_bad_rural]

for word, score in rest_top_words_bad_rural:
    restaurant_dict[word] = -score
    print(word, score)

for word, score in rest_top_words_bad_rural:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Restaurant",
        "review_group": "Bad"
    })

translated google 0.015827375604776747
customer service 0.015145387738793389
food good 0.011318476664306399
ice cream 0.010098253691851692
tasted like 0.009595434758507961
good food 0.007358400967434134
20 minutes 0.006692245202323213
bad food 0.006568875379648311
poor service 0.006441865602468673
terrible service 0.006427882629632192
waste money 0.006101109204691012
don know 0.005682940217135907
30 minutes 0.005562497715261462
extremely rude 0.005499064700825033
waste time 0.005341754273506367
food ok 0.005201786325681828
looked like 0.005148967267278905
chinese food 0.005110671816458276
15 minutes 0.0048421309055863716
10 minutes 0.0047756984210187495


#### Non-Restaurant Bad

In [85]:
state_rural_nonrestaurant_reviews_bad = rural_state_reviews_bad[
    rural_state_reviews_bad['category'].apply(
        lambda cats: not any('restaurant' in c.lower() for c in cats) if isinstance(cats, list) else True
    )]

list_of_rural_state_nonrestaurant_reviews_bad = state_rural_nonrestaurant_reviews_bad.get('text').tolist()

nonrest_rural_tfidf_bad = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=5, ngram_range=(2,2))

nonrest_rural_result_bad = nonrest_rural_tfidf_bad.fit_transform(list_of_rural_state_nonrestaurant_reviews_bad)

nonrest_rural_feature_names_bad = nonrest_rural_tfidf_bad.get_feature_names_out()
nonrest_mean_rural_tfidf_bad = np.asarray(nonrest_rural_result_bad.mean(axis=0)).flatten()
top_n = 20

nonrest_top_indices_bad_rural = nonrest_mean_rural_tfidf_bad.argsort()[-top_n:][::-1]

nonrest_top_words_bad_rural = [(nonrest_rural_feature_names_bad[i], nonrest_mean_rural_tfidf_bad[i]) for i in nonrest_top_indices_bad_rural]

for word, score in nonrest_top_words_bad_rural:
    nonrestaurant_dict[word] = -score
    print(word, score)

for word, score in nonrest_top_words_bad_rural:
    rows.append({
        "bigram": word,
        "score": score,
        "area_type": "Rural",
        "business_type": "Non-Restaurant",
        "review_group": "Bad"
    })

customer service 0.013075454649983512
stay away 0.004599882131329143
don know 0.0039047728542388176
waste time 0.003819552897696281
translated google 0.0037798617389076875
extremely rude 0.002966752596086549
answer phone 0.002880537563130799
zero stars 0.0027541104641529003
phone calls 0.002541388512793257
make sure 0.00249204528329939
worst place 0.002365969599862881
worst experience 0.002326562694082969
poor customer 0.002291858719399125
rude unprofessional 0.0022324668901764436
don care 0.002215194650290081
years ago 0.002199803052745278
year old 0.002089964956387735
terrible service 0.0020782684083253443
don waste 0.002061727968763788
days later 0.002053207755852547


In [86]:
bigram_df = pd.DataFrame(rows)

In [87]:
bigram_df

,bigram,score,area_type,business_type,review_group
0,translated google,0.020587,Urban,Restaurant,Good
1,good food,0.020194,Urban,Restaurant,Good
2,great food,0.019021,Urban,Restaurant,Good
3,food good,0.012148,Urban,Restaurant,Good
4,food great,0.011868,Urban,Restaurant,Good
...,...,...,...,...,...
235,years ago,0.002200,Rural,Non-Restaurant,Bad
236,year old,0.002090,Rural,Non-Restaurant,Bad
237,terrible service,0.002078,Rural,Non-Restaurant,Bad
238,don waste,0.002062,Rural,Non-Restaurant,Bad


In [88]:
bigram_df.to_csv("new_york_insights.csv", index=False)